# 16 | LangGraph Durable Execution：节点失败后从断点恢复

上一篇 Human-in-the-loop 示例展示的是：图主动 `interrupt()`，等待人工审批后再 `Command(resume=...)` 继续。

这一篇换一个恢复场景：**节点运行时失败**。

我们会模拟一个批量摘要管道：

```text
load_items
 -> batch_1
 -> batch_2  失败
 -> 修复 batch_2
 -> 从 batch_2 重新执行
 -> batch_3
 -> export_report
 -> END
```

这个实验要证明一件事：**已经成功 checkpoint 的前置节点不会重跑，失败节点修复后可以从断点继续。**

## 一、准备依赖

这个实验使用 `SqliteSaver`。SQLite 文件会保存在 `data/durable_failure_demo.sqlite`，方便模拟进程重启后的恢复。

批处理节点会真实调用本地 Ollama 模型生成摘要。模型接入方式和上一篇 Human-in-the-loop 实验保持一致。

In [ ]:
from importlib.metadata import version
from operator import add
from pathlib import Path
from typing import Annotated, Any, TypedDict
import sqlite3

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_ollama import ChatOllama
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

print("langgraph", version("langgraph"))
print("langchain-ollama", version("langchain-ollama"))
print("langgraph-checkpoint-sqlite", version("langgraph-checkpoint-sqlite"))


## 二、准备本地模型

这里直接连接本机 Ollama。运行前确认本地已经有模型：

```bash
ollama list
```

如果模型服务没有启动，先启动 Ollama。

In [ ]:
llm = ChatOllama(
    model="qwen3-coder:30b",
    temperature=0,
)

## 三、定义 State

这个 State 刻意保留几类最容易观察的字段：

- `items`：待处理文档；
- `summaries`：每个批次产出的摘要；
- `processed_ids`：已经完成的文档编号；
- `report`：最终报告。

`summaries` 和 `processed_ids` 使用 `operator.add`，表示每个批次返回的新列表会追加到已有列表后面。

In [ ]:
class ReportPipelineState(TypedDict, total=False):
    items: list[str]
    # 使用 add reducer：每个批次的新摘要会追加到已有列表。
    summaries: Annotated[list[str], add]
    processed_ids: Annotated[list[int], add]
    report: str


def require_items(state: ReportPipelineState) -> list[str]:
    items = state.get("items")
    if not isinstance(items, list):
        raise ValueError("State 缺少 items")
    return items


def message_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    return str(content)


def summarize_docs(batch_name: str, docs: list[str]) -> list[str]:
    # 真实调用本地模型，让批处理节点更接近实际业务任务。
    messages = [
        SystemMessage(
            content=(
                "你是一个技术文章摘要助手。"
                "请为每条输入文档各写一句中文摘要。"
                "输出必须每行一条摘要，不要编号，不要解释。"
            )
        ),
        HumanMessage(content="\n".join(docs)),
    ]
    response = llm.invoke(messages)
    lines = [line.strip(" -") for line in message_text(response.content).splitlines() if line.strip()]

    # 如果模型没有严格按“一行一条”返回，就保底生成可观察的摘要。
    if len(lines) != len(docs):
        return [f"{batch_name} 摘要：{doc}" for doc in docs]

    return [f"{batch_name} 摘要：{line}" for line in lines]


## 四、定义稳定节点

`load_items`、`batch_1`、`batch_3` 和 `export_report` 都是稳定节点。后面只替换 `batch_2` 的实现。

In [ ]:
def load_items(state: ReportPipelineState) -> ReportPipelineState:
    print("load_items: 加载 6 条文档")
    return {
        "items": [
            "doc-1: LangGraph 使用 State 在节点之间传递数据。",
            "doc-2: Checkpoint 会保存图执行过程中的状态快照。",
            "doc-3: Durable Execution 可以让任务失败后继续执行。",
            "doc-4: 外部 API 超时是长任务中常见的失败原因。",
            "doc-5: thread_id 用来区分不同的可恢复执行。",
            "doc-6: 生产环境通常会使用 Postgres 等持久化存储。",
        ]
    }


def batch_1(state: ReportPipelineState) -> ReportPipelineState:
    items = require_items(state)
    print("batch_1: 处理 doc-1, doc-2")
    summaries = summarize_docs("batch_1", items[0:2])
    return {"summaries": summaries, "processed_ids": [1, 2]}


def batch_3(state: ReportPipelineState) -> ReportPipelineState:
    items = require_items(state)
    print("batch_3: 处理 doc-5, doc-6")
    summaries = summarize_docs("batch_3", items[4:6])
    return {"summaries": summaries, "processed_ids": [5, 6]}


def export_report(state: ReportPipelineState) -> ReportPipelineState:
    summaries = state.get("summaries", [])
    print("export_report: 生成最终报告")
    return {
        "report": "\n".join(f"{index + 1}. {summary}" for index, summary in enumerate(summaries))
    }

## 五、定义 batch_2 的两个版本

第一次运行使用坏版本，模拟模型服务或外部摘要 API 在 `batch_2` 出现超时。

第二次运行使用修复版本。注意：节点名仍然叫 `batch_2`，只是绑定到这个节点名上的 Python 函数变了。

In [ ]:
def batch_2_broken(state: ReportPipelineState) -> ReportPipelineState:
    print("batch_2: 处理 doc-3, doc-4")
    # 模拟节点在产出结果前失败，因此 batch_2 不会写入 checkpoint。
    raise RuntimeError("模型服务超时，batch_2 失败")


def batch_2_fixed(state: ReportPipelineState) -> ReportPipelineState:
    items = require_items(state)
    print("batch_2: 处理 doc-3, doc-4")
    # 修复后的同名节点会从 checkpoint 记录的位置重新执行。
    summaries = summarize_docs("batch_2", items[2:4])
    return {"summaries": summaries, "processed_ids": [3, 4]}


## 六、封装图构建函数

这里的关键是 `build_app(batch_2_func, db_path)`。

两次构建出来的图结构完全一致：

```text
START -> load_items -> batch_1 -> batch_2 -> batch_3 -> export_report -> END
```

只有 `batch_2` 节点对应的函数不同。

In [ ]:
def build_app(batch_2_func, db_path: Path):
    graph = StateGraph(ReportPipelineState)

    graph.add_node("load_items", load_items)
    graph.add_node("batch_1", batch_1)
    graph.add_node("batch_2", batch_2_func)
    graph.add_node("batch_3", batch_3)
    graph.add_node("export_report", export_report)

    graph.add_edge(START, "load_items")
    graph.add_edge("load_items", "batch_1")
    graph.add_edge("batch_1", "batch_2")
    graph.add_edge("batch_2", "batch_3")
    graph.add_edge("batch_3", "export_report")
    graph.add_edge("export_report", END)

    # 同一个 SQLite 文件 + 同一个 thread_id，才能读到上次失败时的 checkpoint。
    connection = sqlite3.connect(db_path, check_same_thread=False)
    checkpointer = SqliteSaver(connection)
    return graph.compile(checkpointer=checkpointer), connection


## 七、第一次运行：batch_2 失败

为了让实验可以重复运行，先删除旧 SQLite 文件。

`thread_id` 是这次任务的身份。后面恢复时必须继续使用同一个 `thread_id`。

In [ ]:
DB_PATH = Path("data/durable_failure_demo.sqlite")
DB_PATH.parent.mkdir(exist_ok=True)

if DB_PATH.exists():
    # 清理旧实验数据，避免接到上一次运行留下的 checkpoint。
    DB_PATH.unlink()

config = {"configurable": {"thread_id": "failure-recovery-001"}}
config: RunnableConfig = {"configurable": {"thread_id": "failure-recovery-001"}}
app_v1, conn_v1 = build_app(batch_2_broken, DB_PATH)

try:
    app_v1.invoke({"summaries": [], "processed_ids": []}, config=config)
except RuntimeError as exc:
    print(f"第一次运行失败：{exc}")

snapshot = app_v1.get_state(config)

print("\ncheckpoint 中保存的下一步：", snapshot.next)
print("checkpoint 中保存的 processed_ids：", snapshot.values.get("processed_ids"))
print("checkpoint 中保存的 summaries 数量：", len(snapshot.values.get("summaries", [])))

conn_v1.close()


这说明 `batch_1` 的结果已经保存，但 `batch_2` 没有成功完成。

## 八、第二次运行：修复 batch_2 后继续

现在重新构建图，把 `batch_2` 节点绑定到修复后的 `batch_2_fixed`。

注意这里继续使用同一个 SQLite 文件和同一个 `thread_id`，并且输入是 `None`。`None` 表示不从入口提供新输入，而是继续 checkpoint 中记录的未完成节点。

In [ ]:
app_v2, conn_v2 = build_app(batch_2_fixed, DB_PATH)

# None 表示不提供新的入口输入，而是继续 checkpoint 里的 next 节点。
result = app_v2.invoke(None, config=config)

print("\n最终 processed_ids：", result.get("processed_ids"))
print("最终 summaries 数量：", len(result.get("summaries", [])))
print("\n最终报告：")
print(result.get("report"))

conn_v2.close()


第二次运行，注意没有再次打印：

```text
load_items: 加载 6 条文档
batch_1: 处理 doc-1, doc-2
```

这就是这个实验最重要的观察结果：**前面已经成功 checkpoint 的节点不会重跑，图从 checkpoint 记录的 `batch_2` 继续。**

## 九、查看最终状态

再重新打开一次 SQLite 连接，模拟服务重启后读取最终 checkpoint。

In [ ]:
app_final, conn_final = build_app(batch_2_fixed, DB_PATH)
final_snapshot = app_final.get_state(config)

print("下一步节点：", final_snapshot.next)
print("最终 processed_ids：", final_snapshot.values.get("processed_ids"))
print("最终 report 是否存在：", "report" in final_snapshot.values)

conn_final.close()

完成后，`next` 应该是空元组 `()`，表示图已经走到 `END`。

## 十、和 Human-in-the-loop 示例的区别

| 对比点 | Human-in-the-loop 恢复 | 节点失败后恢复 |
| --- | --- | --- |
| 共同前提 | 都需要 `compile(checkpointer=...)` | 都需要 `compile(checkpointer=...)` |
| 执行身份 | 继续使用同一个 `thread_id` | 继续使用同一个 `thread_id` |
| 中断原因 | 业务上主动等待人工审批 | 节点运行失败 |
| 停止方式 | `interrupt(...)` | `raise` / 异常 |
| 除 checkpoint 外还需要什么 | `interrupt(...)` 制造暂停点，`Command(resume=...)` 带回人工输入 | 修复失败节点，或等待外部故障恢复 |
| 恢复调用 | `invoke(Command(resume=...), config)` | `invoke(None, config)` |
| 恢复位置 | 回到 `interrupt` 暂停点，并把 `resume` 内容作为 `interrupt(...)` 的返回值 | 重新执行失败节点，然后继续后续节点 |
| 已完成节点 | 不会重新调用审批前已完成的节点 | 不会重新调用失败前已 checkpoint 的节点 |
| 典型场景 | 审批、审核、人工补充信息 | 长任务、批处理、外部 API 不稳定 |
| 工程边界 | 人工输入要作为恢复数据进入 State | 失败节点内部已经发生的外部副作用不会自动回滚 |

所以，`checkpointer` 是恢复能力的前提，但不是全部。Human-in-the-loop 还需要 `interrupt(...)` 和 `Command(resume=...)`；节点失败后恢复还需要同一个 `thread_id`、修复后的节点实现，以及用 `invoke(None, config)` 继续 checkpoint 中记录的未完成节点。


## 十一、实验结论

这个实验可以总结为三句话：

- Durable Execution 的恢复边界通常是节点；
- 已经成功执行并写入 checkpoint 的节点不会重复执行；
- 失败节点修复后会重新执行，然后继续跑后续节点直到 `END`。

所以在设计长任务时，一个很实用的原则是：**把稳定阶段拆成节点，让节点边界成为可恢复边界。**